In [2]:
import os
os.chdir("..")

In [3]:
import asyncio
from datetime import datetime, timedelta
from typing import Optional
from pprint import pprint

# Jupyter + asyncio
import nest_asyncio
nest_asyncio.apply()

In [4]:
from sqlalchemy.ext.asyncio import AsyncSession

from src.database.connection import async_session
from src.database.schema import Article, Event, Weather
from src.ai.summary_generator import SummaryGenerator


In [57]:
async def inspect_summary_input(
    target_date: Optional[datetime] = None
):
    generator = SummaryGenerator()

    async with async_session() as session:
        # --- dokładnie to samo co w generate_daily_summary ---
        if target_date is None:
            target_date = (
                datetime.utcnow()
                .replace(hour=0, minute=0, second=0, microsecond=0)- timedelta(days=1)
            )

        date_start = target_date
        date_end = date_start + timedelta(days=2)

        # Artykuły
        articles_result = await session.execute(
            select(Article)
            .where(Article.published_at >= date_start)
            .where(Article.published_at < date_end)
            .where(Article.processed == True)
            .order_by(Article.published_at.desc())
        )
        articles = articles_result.scalars().all()

        articles_by_category = {}
        for article in articles:
            category = article.category or "Inne"
            articles_by_category.setdefault(category, []).append(article)

        # Wydarzenia
        events_result = await session.execute(
            select(Event)
            .where(Event.event_date >= date_end)
            .where(Event.event_date < date_end + timedelta(days=7))
            .order_by(Event.event_date.asc())
            .limit(10)
        )
        events = events_result.scalars().all()

        # Pogoda
        weather_result = await session.execute(
            select(Weather)
            .where(Weather.is_current == True)
            .order_by(Weather.fetched_at.desc())
            .limit(1)
        )
        weather = weather_result.scalar_one_or_none()

        # INPUT DLA AI (TO NAS INTERESUJE)
        input_text = generator._prepare_input_for_ai(
            date_start,
            articles_by_category,
            events,
            weather
        )

        return {
            "date": date_start,
            "articles_count": len(articles),
            "categories": list(articles_by_category.keys()),
            "events_count": len(events),
            "weather": weather,
            "input_text": input_text
        }


In [58]:
data = await inspect_summary_input()

print("DATA:", data["date"])
print("LICZBA ARTYKUŁÓW:", data["articles_count"])
print("KATEGORIE:", data["categories"])
print("LICZBA WYDARZEŃ:", data["events_count"])


DATA: 2026-01-11 00:00:00
LICZBA ARTYKUŁÓW: 11
KATEGORIE: ['Kultura', 'Rekreacja', 'Nieruchomości', 'Urząd', 'Edukacja']
LICZBA WYDARZEŃ: 10


In [59]:
print(data["input_text"])


Data: 2026-01-11
Liczba artykułów: 11

ARTYKUŁY PO KATEGORIACH:


## EDUKACJA (2 artykułów):

1. Inwestycja w edukację! 🎓💻
   → Nowe komputery i tablety zostały wprowadzone do szkół w celu wspierania kreatywności oraz rozwoju cyfrowych kompetencji uczniów. Inwestycja z pewnością przyczyni się do innowacyjności w edukacji.

2. Ślubowanie klas OPW oraz humanistycznej
   → Ślubowanie klas OPW i humanistycznej to ważny moment integrujący społeczność szkolną oraz budujący poczucie przynależności. To wydarzenie ma na celu życzenie uczniom sukcesów, wytrwałości i inspirujących doświadczeń w trakcie ich nauki.


## KULTURA (6 artykułów):

1. SiS Rybno i Rumian w dzisiejszej kolejce w Lidze Halowej w Byszwałdzie o Puchar Wójta Gminy Lubawa s...
   → W Lidze Halowej odbyła się kolejka, w której uczestniczyły drużyny SiS Rybno oraz Rumian. W meczach tej kolejki strzelono 38 bramek, a szczególnie wyróżnił się zespół SiS Rybno, który wygrał z Kajmanem Lubawa 15:0.
   📍 Rybno

2. 🙏 Dziękujemy organi